
# NB_OTT_GenerateIncidentSummaries

 Objetivo:
Generar una descripción operacional para cada grupo de
tickets detectado previamente por el clustering.

Entrada:
* Gold.IncidentGroups

Salida:
* Gold.IncidentSummaries

Este notebook forma parte del pipeline operacional.

 IMPORTANTE:
* GenAI NO realiza el clustering.
* GenAI interpreta los grupos producidos aguas arriba.
* El prompt indica explícitamente que un cluster puede contener problemas heterogéneos.


In [1]:
# Número máximo de tickets utilizados como contexto para
# generar el resumen de cada cluster.
max_tickets_genai = 20

# Modelo utilizado para la generación.
genai_model = "gpt-5.1"

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 5, Finished, Available, Finished, False)

In [2]:
# ============================================================
# 1. Imports y configuración
#
# Las dependencias están gestionadas mediante el Environment ENV_OTT_TicketIntelligence. No se realizan instalaciones dinámicas dentro del notebook.
# ============================================================

import json
import random

import pandas as pd
import openai

from pyspark.sql import functions as F
from synapse.ml.fabric.credentials import (
    get_openai_httpx_sync_client
)


MAX_TICKETS = int(max_tickets_genai)
GENAI_MODEL = str(genai_model)

# Semilla fija para que el muestreo de tickets sea reproducible entre ejecuciones.
RANDOM_SEED = 42

print("GenAI configuration")
print("-------------------")
print("Model:", GENAI_MODEL)
print("Max tickets per cluster:", MAX_TICKETS)
print("Random seed:", RANDOM_SEED)

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 6, Finished, Available, Finished, False)

GenAI configuration
-------------------
Model: gpt-5.1
Max tickets per cluster: 20
Random seed: 42


In [3]:
# ============================================================
# 2. Carga de grupos de incidentes
#
# Gold.IncidentGroups contiene un registro por cluster con la información agregada necesaria para la generación.
# ============================================================

df_groups = spark.table(
    "Gold.IncidentGroups"
)

group_count = df_groups.count()

assert group_count > 0, (
    "Gold.IncidentGroups does not contain any groups."
)

groups_pdf = df_groups.toPandas()

print(
    f"Incident groups available: {group_count}"
)

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 7, Finished, Available, Finished, False)

Incident groups available: 14


In [4]:
# ============================================================
# 3. Prompt de sistema
#
# El modelo actúa como capa de síntesis de los tickets.
#
# Se le exige:
# - trabajar únicamente con la información suministrada
# - no asumir una causa raíz común
# - no inventar datos técnicos
# - advertir cuando el cluster parezca heterogéneo
# - devolver únicamente JSON estructurado.
# ============================================================

SYSTEM_PROMPT = """
You are an assistant supporting the operation of an OTT platform.

You receive support tickets that have been automatically grouped
using semantic embeddings and clustering.

Your task is to summarize the operational information contained
in the supplied tickets.

Important rules:
- Base your conclusions only on the supplied tickets.
- Do not invent technical information.
- Do not assume that all tickets necessarily have the same root cause.
- Distinguish observations from probable interpretations.
- Recommended actions must be reasonable investigation or operational
  next steps based only on the available evidence.
- If the cluster appears to contain multiple different problems,
  explicitly indicate this in the warning.
- If the evidence is insufficient to identify a probable issue,
  say so explicitly.

Return ONLY valid JSON with exactly this structure:

{
  "generated_title": "...",
  "generated_summary": "...",
  "probable_issue": "...",
  "affected_scope": "...",
  "recommended_action": "...",
  "warning": "..."
}
"""

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 8, Finished, Available, Finished, False)

In [5]:
# ============================================================
# 4. Cliente Azure OpenAI integrado con Microsoft Fabric
# ============================================================

client = openai.AzureOpenAI(
    http_client=get_openai_httpx_sync_client(),
    api_version="2025-04-01-preview"
)

print("Azure OpenAI client initialized successfully.")

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 9, Finished, Available, Finished, False)

Azure OpenAI client initialized successfully.


In [6]:
# ============================================================
# 5. Selección reproducible de tickets representativos
#
# Los clusters pueden contener más tickets de los que resulta práctico incluir en una única llamada GenAI.
#
# Para el TFM se utiliza un muestreo aleatorio reproducible. La semilla fija permite repetir la ejecución utilizando el mismo subconjunto de tickets.
#
# Una evolución futura podría utilizar selección basada en:
# - proximidad al centroide,
# - diversidad semántica,
# - prioridad,
# - error_code,
# - cobertura de plataformas/servicios.
# ============================================================

def sample_ticket_texts(
    ticket_texts,
    max_tickets,
    seed=42
):
    if ticket_texts is None:
        return []

    ticket_texts = list(ticket_texts)

    # Eliminamos elementos vacíos.
    ticket_texts = [
        str(text).strip()
        for text in ticket_texts
        if text is not None
        and str(text).strip()
    ]

    if len(ticket_texts) <= max_tickets:
        return ticket_texts

    rng = random.Random(seed)

    return rng.sample(
        ticket_texts,
        max_tickets
    )

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 10, Finished, Available, Finished, False)

In [7]:
# ============================================================
# 6. Generación de un resumen para un cluster
#
# La función:
# 1. selecciona el contexto
# 2. construye el prompt
# 3. invoca el modelo
# 4. valida el JSON
# 5. devuelve una estructura preparada para Gold.
# ============================================================

EXPECTED_FIELDS = [
    "generated_title",
    "generated_summary",
    "probable_issue",
    "affected_scope",
    "recommended_action",
    "warning"
]


def generate_incident_summary(
    row,
    client,
    model_name,
    max_tickets
):
    cluster_id = int(
        row["cluster_id"]
    )

    ticket_count = int(
        row["ticket_count"]
    )

    ticket_texts = (
        row["ticket_texts"]
    )

    # ----------------------------------------
    # Selección del contexto
    # ----------------------------------------

    sampled_tickets = sample_ticket_texts(
        ticket_texts=ticket_texts,
        max_tickets=max_tickets,
        seed=RANDOM_SEED + cluster_id
    )

    if len(sampled_tickets) == 0:
        raise ValueError(
            f"Cluster {cluster_id} contains no valid ticket texts."
        )

    tickets_text = "\n\n".join(
        f"Ticket {i + 1}: {text}"
        for i, text in enumerate(
            sampled_tickets
        )
    )

    # ----------------------------------------
    # Prompt específico del cluster
    # ----------------------------------------

    user_prompt = f"""
Cluster ID: {cluster_id}
Total number of tickets in cluster: {ticket_count}
Number of tickets supplied as context: {len(sampled_tickets)}

Tickets analyzed:
{tickets_text}
"""

    # ----------------------------------------
    # Invocación GenAI
    # ----------------------------------------

    response = (
        client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )
    )

    result_text = (
        response
        .choices[0]
        .message
        .content
    )

    if not result_text:
        raise ValueError(
            f"Empty response for cluster {cluster_id}."
        )

    # ----------------------------------------
    # Validación del JSON
    # ----------------------------------------

    result = json.loads(
        result_text
    )

    missing_fields = [
        field
        for field in EXPECTED_FIELDS
        if field not in result
    ]

    if missing_fields:
        raise ValueError(
            f"Missing fields in GenAI response: "
            f"{missing_fields}"
        )

    # ----------------------------------------
    # Resultado normalizado
    # ----------------------------------------

    return {
        "cluster_id":
            cluster_id,

        "ticket_count":
            ticket_count,

        "tickets_used_for_context":
            len(sampled_tickets),

        "generated_title":
            result.get(
                "generated_title"
            ),

        "generated_summary":
            result.get(
                "generated_summary"
            ),

        "probable_issue":
            result.get(
                "probable_issue"
            ),

        "affected_scope":
            result.get(
                "affected_scope"
            ),

        "recommended_action":
            result.get(
                "recommended_action"
            ),

        "warning":
            result.get(
                "warning"
            )
    }

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 11, Finished, Available, Finished, False)

In [8]:
# ============================================================
# 7. Procesamiento de todos los grupos
#
# Cada cluster se procesa de forma independiente.
#
# Un error en una llamada GenAI no interrumpe necesariamente el procesamiento de los demás grupos.
#
# El estado del resultado se persiste para permitir trazabilidad y diagnóstico posterior.
# ============================================================

results = []

for _, row in groups_pdf.iterrows():

    cluster_id = int(
        row["cluster_id"]
    )

    try:

        result = (
            generate_incident_summary(
                row=row,
                client=client,
                model_name=GENAI_MODEL,
                max_tickets=MAX_TICKETS
            )
        )

        result[
            "generation_status"
        ] = "success"

        result[
            "generation_error"
        ] = None

        print(
            f"Cluster {cluster_id}: success"
        )

    except Exception as e:

        result = {
            "cluster_id":
                cluster_id,

            "ticket_count":
                int(
                    row["ticket_count"]
                ),

            "tickets_used_for_context":
                None,

            "generated_title":
                None,

            "generated_summary":
                None,

            "probable_issue":
                None,

            "affected_scope":
                None,

            "recommended_action":
                None,

            "warning":
                None,

            "generation_status":
                "failed",

            "generation_error":
                str(e)[:1000]
        }

        print(
            f"Cluster {cluster_id}: failed - {e}"
        )

    results.append(
        result
    )

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 12, Finished, Available, Finished, False)

Cluster 0: success
Cluster 7: success
Cluster 6: success
Cluster 9: success
Cluster 5: success
Cluster 1: success
Cluster 10: success
Cluster 3: success
Cluster 12: success
Cluster 8: success
Cluster 11: success
Cluster 2: success
Cluster 4: success
Cluster 13: success


In [9]:
# ============================================================
# 7. Procesamiento de todos los grupos
#
# Cada cluster se procesa de forma independiente.
#
# Un error en una llamada GenAI no interrumpe necesariamente el procesamiento de los demás grupos.
#
# El estado del resultado se persiste para permitir trazabilidad y diagnóstico posterior.
# ============================================================

results = []

for _, row in groups_pdf.iterrows():

    cluster_id = int(
        row["cluster_id"]
    )

    try:

        result = (
            generate_incident_summary(
                row=row,
                client=client,
                model_name=GENAI_MODEL,
                max_tickets=MAX_TICKETS
            )
        )

        result[
            "generation_status"
        ] = "success"

        result[
            "generation_error"
        ] = None

        print(
            f"Cluster {cluster_id}: success"
        )

    except Exception as e:

        result = {
            "cluster_id":
                cluster_id,

            "ticket_count":
                int(
                    row["ticket_count"]
                ),

            "tickets_used_for_context":
                None,

            "generated_title":
                None,

            "generated_summary":
                None,

            "probable_issue":
                None,

            "affected_scope":
                None,

            "recommended_action":
                None,

            "warning":
                None,

            "generation_status":
                "failed",

            "generation_error":
                str(e)[:1000]
        }

        print(
            f"Cluster {cluster_id}: failed - {e}"
        )

    results.append(
        result
    )

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 13, Finished, Available, Finished, False)

Cluster 0: success
Cluster 7: success
Cluster 6: success
Cluster 9: success
Cluster 5: success
Cluster 1: success
Cluster 10: success
Cluster 3: success
Cluster 12: success
Cluster 8: success
Cluster 11: success
Cluster 2: success
Cluster 4: success
Cluster 13: success


In [10]:
# ============================================================
# 8. Preparación de resultados
#
# Además del contenido generado, se persisten metadatos que permiten conocer:
#
# - modelo utilizado
# - límite de contexto
# - resultado de la generación
# - momento de ejecución
# ============================================================

summaries_pdf = pd.DataFrame(
    results
)

df_summaries = (
    spark.createDataFrame(
        summaries_pdf
    )
    .withColumn(
        "genai_model",
        F.lit(
            GENAI_MODEL
        )
    )
    .withColumn(
        "max_tickets_context",
        F.lit(
            MAX_TICKETS
        )
    )
    .withColumn(
        "_generated_at",
        F.current_timestamp()
    )
)

print(
    "Summaries prepared:",
    df_summaries.count()
)

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 14, Finished, Available, Finished, False)

Summaries prepared: 14


In [11]:
# ============================================================
# 9. Persistencia en Gold
#
# Gold.IncidentSummaries contiene un registro por cluster.
#
# Utilizamos overwrite para mantener una ejecución reproducible del pipeline completo.
#
# En un escenario productivo esta estrategia podría evolucionar hacia MERGE/incremental processing.
# ============================================================

(
    df_summaries.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        "Gold.IncidentSummaries"
    )
)

print(
    "Gold.IncidentSummaries written successfully."
)

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 15, Finished, Available, Finished, False)

Gold.IncidentSummaries written successfully.


In [12]:
# ============================================================
# 10. Data Quality / Pipeline validation
#
# Verificamos que exista un resultado por cada IncidentGroup. También reportamos cuántas llamadas finalizaron correctamente y cuántas fallaron.
# ============================================================

df_saved = spark.table(
    "Gold.IncidentSummaries"
)

saved_count = (
    df_saved.count()
)

success_count = (
    df_saved
    .filter(
        F.col(
            "generation_status"
        ) == "success"
    )
    .count()
)

failed_count = (
    df_saved
    .filter(
        F.col(
            "generation_status"
        ) == "failed"
    )
    .count()
)


assert saved_count == group_count, (
    f"Expected {group_count} incident summaries/results "
    f"but found {saved_count}."
)


print("GenAI generation validation")
print("---------------------------")
print(
    "Incident groups:",
    group_count
)

print(
    "Results stored:",
    saved_count
)

print(
    "Successful generations:",
    success_count
)

print(
    "Failed generations:",
    failed_count
)

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 16, Finished, Available, Finished, False)

GenAI generation validation
---------------------------
Incident groups: 14
Results stored: 14
Successful generations: 14
Failed generations: 0


In [13]:
assert failed_count == 0, (
    f"{failed_count} GenAI generations failed."
)

print(
    "\nNB_OTT_GenerateIncidentSummaries "
    "completed successfully."
)

StatementMeta(, d9cc311a-6120-4595-bfc8-f8b204c15c6a, 17, Finished, Available, Finished, False)


NB_OTT_GenerateIncidentSummaries completed successfully.
